![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 05: Knowledge Agents and Stateful Workflows)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-AI-lab](https://github.com/tulip-lab/agentic-AI-lab/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 5A: Basic RAG System

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Local RAG system using approved public snippets and lexical retrieval</td></tr>
<tr><td align="left">Optional part</td><td>Embedding retrieval or real model answer generation if packages and API key are available</td></tr>
<tr><td align="left">Main output</td><td>A small RAG pipeline with retrieval, grounded answer construction, source display, limitations and tests</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m05a-overview)
2. [Setup and Background](#m05a-background)
3. [Core Concepts](#m05a-knowledge-base)
4. [Guided Implementation](#m05a-local-rag)
5. [Testing and Analysis](#m05a-testing)
6. [Student Tasks](#m05a-student-tasks)
7. [Submission and Reflection](#m05a-submission)

---

<a id="m05a-overview"></a>

### 1. Overview and Learning Goals

M05 starts the knowledge-agent part of the unit. In M02C, you built transparent lexical retrieval and examined how embeddings could replace its scoring stage. In M03C, you built a visual RAG workflow in Flowise. In M04, you implemented prompt, parser, tool and controlled-action patterns in Python. M05A now builds a small RAG system directly in code, so that every stage Flowise previously drew for you becomes something you can read, run and test.

RAG means **retrieval-augmented generation**. A RAG system does not answer only from the model's internal memory. It first retrieves relevant context from an approved knowledge base, and then constructs an answer using only that retrieved context.

A useful analogy is an open-book exam. A model answering on its own is like a student answering from memory: fluent, confident, and sometimes wrong. A RAG system is like a student in an open-book exam who must first find the right page and is only allowed to write down what that page supports. If the book does not cover the question, the honest response is "the material does not contain this", not a confident guess.

The full pipeline you will build looks like this:

```text
                  +--------------------------+
                  | Approved knowledge base  |
                  +------------+-------------+
                               |
                               v
User question ----------> [ Retriever ]
                               |
                               v
                       Retrieved context
                               |
                               v
                  [ Grounded answer builder ]
                               |
                               v
                Answer + sources + limitations
```

This notebook uses a mandatory local RAG pipeline. It does not require an API key. It uses approved public snippets and a simple lexical retriever. The purpose is to make the RAG architecture visible and testable before you move on to embeddings, vector databases or real model calls.

By the end of this session, you should be able to:

```text
1. Explain the difference between retrieval and generation.
2. Build a small local retriever.
3. Construct an answer only from retrieved context.
4. Display sources with the answer.
5. Detect insufficient context.
6. Test normal, weak-evidence and invalid-input cases.
7. Explain why RAG systems need retrieval inspection and source grounding.
```


<a id="m05a-background"></a>

### 2. Setup and Background

#### 2.1 What RAG Solves

#### 2.1 The problem with answering from the model alone

A language model can often produce a fluent answer even when it does not have the right evidence. This is useful for general writing, but risky for knowledge-intensive systems. In a unit assistant, a research assistant, a policy assistant or a technical support assistant, the answer should be based on specific approved information.

A model-only workflow is:

```text
Question --> Prompt --> Model --> Answer
             (there is no evidence step anywhere)
```

The weakness is that the answer may not be grounded in the material you want the system to use. It may sound plausible while inventing details, and you have no way to check it, because nothing in the pipeline records what the answer was based on.

A RAG workflow adds an evidence step:

```text
                 +----------------+
                 | Knowledge base |
                 +-------+--------+
                         |
Question --> [ Retrieve relevant context ]
                         |
                         v
           Prompt that contains the context
                         |
                         v
            Model or local answer builder
                         |
                         v
           Grounded answer + sources
```

The model is no longer asked to answer from nowhere. It is asked to answer from retrieved evidence, and the evidence itself is visible for inspection.

#### 2.2 Retrieval is not generation

Retrieval and generation are different jobs, and they fail in different ways. When you debug a RAG system, the first question should always be: which of the two stages went wrong?

<div align="center">

<table>
<thead>
<tr><th><strong>Stage</strong></th><th><strong>Question it answers</strong></th><th><strong>Typical failure</strong></th><th><strong>What to inspect</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Retrieval</td><td>Which documents are relevant?</td><td>Wrong or missing context.</td><td>Retrieved documents, scores, source titles.</td></tr>
<tr><td align="left">Generation</td><td>How should the answer be written?</td><td>Unsupported or over-confident answer.</td><td>Whether the answer is supported by retrieved context.</td></tr>
</tbody>
</table>

</div>

A RAG system can fail even if the final answer sounds good. For example, the retriever may return the wrong document, and the answer builder may still write a confident answer. Therefore, in this notebook, every answer will display the retrieved sources and limitations.

#### 2.3 What does "grounded" mean?

An answer is grounded when its claims can be traced back to retrieved context. If the retrieved documents do not say something, the answer should not claim it. If the retrieved context is weak or missing, the system should say that the available context is insufficient.

Good RAG behaviour:

```text
Question: What does M05A teach?
Retrieved source: M05A snippet about retrieval and grounding.
Answer: M05A teaches local RAG, retrieval, sources and insufficient-context handling.
```

Bad RAG behaviour:

```text
Question: What is the final exam room?
Retrieved source: no relevant document.
Answer: The final exam is in Room 201.
```

The second answer is unacceptable because it invents information not contained in the approved context. In the open-book exam analogy, this is a student writing an answer while pointing at a blank page.

#### 2.4 How this connects to previous modules

<div align="center">

<table>
<thead>
<tr><th><strong>Previous session</strong></th><th><strong>Concept carried into M05A</strong></th><th><strong>How it appears here</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">M02C</td><td>Similarity and representation.</td><td>We begin with lexical similarity and later compare this with embeddings.</td></tr>
<tr><td align="left">M03C</td><td>Visual RAG in Flowise.</td><td>The same RAG pipeline is now implemented in code.</td></tr>
<tr><td align="left">M04A</td><td>Prompt/model/parser pipeline.</td><td>The answer builder plays the role of a controlled generation component.</td></tr>
<tr><td align="left">M04D</td><td>Evidence-grounded drafting.</td><td>Answers must show evidence and limitations.</td></tr>
</tbody>
</table>

</div>


<a id="m05a-setup"></a>

#### 2.2 Environment and Safety

This notebook uses only standard Python for the mandatory section. It does not require a real API key, no packages need to be installed, and every cell runs the same way in Google Colab and local Jupyter. The optional section later shows what a real model call would look like, but the required learning outcome is the local RAG pipeline.

The mandatory pipeline is:

```text
approved documents -> lexical retriever -> retrieved context
                   -> conservative answer builder -> answer with sources
```

Run the setup cell below. You should see `M05A setup complete.` and nothing else. If you see an error instead, restart the runtime and run the cell again before continuing.

Use only approved public or synthetic teaching material in this notebook. Do not index private documents, student submissions, hidden instructor solutions, credentials, emails or personal records. A RAG system exposes whatever you index: if a private document enters the knowledge base, its content can appear in answers.


In [ ]:
# The mandatory pipeline needs only the Python standard library.
# This is a design decision: with no external packages and no API keys,
# every student runs the same pipeline, gets the same results, and can
# inspect every stage of the RAG architecture directly.

import json  # pretty-printing documents and structured results
import re    # tokenising text for the lexical retriever
from typing import Any, Dict, List

print("M05A setup complete.")


<a id="m05a-knowledge-base"></a>

### 3. Core Concepts

#### 3.1 Approved Knowledge Base

The knowledge base below simulates public unit material. Each document is short so that you can inspect retrieval behaviour manually: when a query matches, you can read the whole document and explain exactly why it was chosen.

Each document has:

```text
doc_id: a stable identifier used in source citations
title:  short title shown with retrieval results
text:   the approved content answers may be built from
source: where the content came from (so readers can verify it)
tags:   topic labels that give the lexical retriever extra matching terms
```

The `tags` field matters more than it looks. The retriever in this notebook matches words, so a document about RAG that never contains the word "retrieval" would be hard to find with a retrieval question. Tags let you add matching vocabulary without rewriting the text.

A real RAG system may index PDFs, web pages, Markdown files, notebooks or databases. The same principle applies at any scale: only approved documents should be indexed. Run the next cell and check that it reports five documents and prints the first one in full.


In [ ]:
# The knowledge base is deliberately tiny: five short, approved snippets.
# Small documents make retrieval transparent. In production you would index
# many chunked documents, but the approval rule is identical: never index
# private or unapproved material, even in a demo.

KNOWLEDGE_BASE = [
    {
        "doc_id": "D001",
        "title": "Flowise and Visual Workflows",
        "text": "Flowise represents AI workflows visually. Students connect nodes such as prompts, chat models, retrievers, tools and outputs to understand information flow.",
        "source": "public unit snippet",
        "tags": ["flowise", "visual_workflow", "nodes", "tools"]
    },
    {
        "doc_id": "D002",
        "title": "LangChain Code Workflows",
        "text": "LangChain-style workflows express prompt templates, model calls, parsers and chains in Python code. This makes AI workflows testable and reusable.",
        "source": "public unit snippet",
        "tags": ["langchain", "prompt", "parser", "chain"]
    },
    {
        "doc_id": "D003",
        "title": "RAG Fundamentals",
        "text": "RAG retrieves relevant context before constructing an answer. A good RAG system should show sources and avoid answering beyond the retrieved evidence.",
        "source": "public unit snippet",
        "tags": ["rag", "retrieval", "sources", "grounding"]
    },
    {
        "doc_id": "D004",
        "title": "Tool Agent Safety",
        "text": "Tool agents can call approved functions, but they need validation and refusal rules. Unsafe tools such as shell commands or private-file access should be excluded from early labs.",
        "source": "public unit snippet",
        "tags": ["tool_agents", "safety", "validation", "refusal"]
    },
    {
        "doc_id": "D005",
        "title": "Stateful Workflows",
        "text": "Stateful workflows keep track of decisions, intermediate outputs and transitions. LangGraph can represent branching workflows such as success, validation error and refusal.",
        "source": "public unit snippet",
        "tags": ["langgraph", "state", "workflow", "branching"]
    },
]

print("Number of documents:", len(KNOWLEDGE_BASE))
print(json.dumps(KNOWLEDGE_BASE[0], indent=2))

#### 4.1 Why these documents are small

Small snippets are easier to inspect. When you run a query, you can quickly see whether the right document was retrieved. This is valuable for learning because RAG quality depends strongly on retrieval quality.

In a production RAG system, documents are often split into chunks. Each chunk should be large enough to preserve meaning but small enough to retrieve precisely. This notebook does not implement full chunking yet; M05B will move closer to unit-material RAG.

<a id="m05a-local-rag"></a>

### 4. Guided Implementation

#### 4.1 Mandatory Local RAG Pipeline

#### 5.1 Tokenisation and lexical scoring

The local retriever needs a way to compare the question with each document. We start with the simplest method that works: split both texts into lowercase terms and count how many terms they share.

One refinement is needed straight away. Words such as "the", "is" and "what" appear in almost every English sentence, so counting them as evidence would make every document look slightly relevant to every question — and a question like "What is the final exam room?" would wrongly retrieve a document just because both contain "the". The tokeniser therefore removes these **stopwords**, so that a score of zero really means "no meaningful overlap".

This is called lexical retrieval because it depends on the exact words used. It is not semantic retrieval: it has no idea that "car" and "automobile" mean the same thing. If the question uses very different words from the document, lexical retrieval will miss relevant material even when a human would consider it obviously related. Observing this limitation now is exactly what motivates embeddings later — you will know what problem they solve.

Run the tokeniser cell below first. You should see a list of lowercase content words, with punctuation and stopwords removed. If tokenisation looks wrong, everything downstream will look wrong too, so it is worth checking this smallest piece on its own.


In [ ]:
# Stopwords are common words that carry almost no topical meaning.
# Counting them as evidence would let every document match every question,
# because "the" or "is" appears everywhere. Removing them keeps a score of
# zero meaningful: it really indicates no topical overlap.
STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "to", "in", "on", "for", "at",
    "is", "are", "was", "were", "be", "been", "it", "its", "this", "that",
    "what", "which", "how", "why", "when", "where", "who",
    "do", "does", "did", "should", "can", "could", "will", "would",
    "with", "from", "by", "as", "they", "their", "you", "your",
}


def tokenize(text: str) -> List[str]:
    """Split text into lowercase content-word tokens."""
    # Lowercasing makes matching case-insensitive: "RAG" in a question
    # still matches "rag" in a document.
    # The regex keeps only alphabetic words (plus underscores used in tags),
    # discarding punctuation and digits that would add matching noise.
    # Returning [] for non-string input keeps later stages simple: they can
    # always assume they receive a list, never an exception.
    if not isinstance(text, str):
        return []
    words = re.findall(r"[a-zA-Z_]+", text.lower())
    return [word for word in words if word not in STOPWORDS]


# Expected output: lowercase content words only -- no punctuation,
# and no stopwords such as "the" or "is".
print(tokenize("RAG retrieves relevant context before answering."))
print(tokenize("What is the final exam room?"))


In [ ]:
def score_document(query: str, document: Dict[str, Any]) -> int:
    """Score a document by keyword overlap with the query."""

    # Two design decisions worth noticing:
    # 1. Title, text and tags are merged into one searchable string, so a
    #    query can match a document through any of the three fields.
    # 2. Sets are used, so repeating a word in the query does not inflate
    #    the score: it counts *distinct* shared terms.
    query_terms = set(tokenize(query))
    doc_text = " ".join([
        document.get("title", ""),
        document.get("text", ""),
        " ".join(document.get("tags", [])),
    ])
    doc_terms = set(tokenize(doc_text))

    return len(query_terms.intersection(doc_terms))


# D003 (RAG Fundamentals) should score highest for this query.
# If a different document wins, read its tags to see which terms matched.
for doc in KNOWLEDGE_BASE:
    print(doc["doc_id"], doc["title"], "score=", score_document("How does RAG use retrieved sources?", doc))


The score is simple: more overlapping content words means a higher score. This is not the strongest retrieval method, but it is completely transparent — for any retrieved document, you can list the exact shared words that caused the match. Being able to explain *why* a document was retrieved is the habit this section is building; it applies unchanged when the scores later come from embedding similarity.

Try removing a word from the query and re-running the cell to watch the scores change. If a score surprises you, print the two token sets and inspect their intersection — the evidence is always inspectable.


In [ ]:
def retrieve_documents(query: str, knowledge_base: List[Dict[str, Any]], top_k: int = 3) -> Dict[str, Any]:
    """Retrieve top-k documents using lexical overlap."""

    # Validate inputs first, and report problems through the same
    # ok/error/result envelope used throughout M04. A retriever that
    # crashes on bad input is much harder to embed in a larger workflow
    # than one that returns a clear, machine-readable error.
    if not isinstance(query, str) or not query.strip():
        return {"ok": False, "error": "query must be a non-empty string.", "result": None}

    if not isinstance(top_k, int) or top_k <= 0:
        return {"ok": False, "error": "top_k must be a positive integer.", "result": None}

    # Keep only documents that share at least one term with the query.
    # This filter is why an off-topic question returns an empty list,
    # which the answer builder later turns into an honest
    # "insufficient context" reply instead of an invented answer.
    scored = []
    for doc in knowledge_base:
        score = score_document(query, doc)
        if score > 0:
            scored.append((score, doc))

    # Best evidence first. top_k = 3 is a sensible default for this small
    # knowledge base: enough context to answer, few enough results to
    # inspect by eye. Larger top_k values add weaker, noisier evidence.
    scored.sort(key=lambda pair: pair[0], reverse=True)

    results = []
    for score, doc in scored[:top_k]:
        item = dict(doc)          # copy, so the knowledge base itself is never mutated
        item["score"] = score     # keep the score visible for inspection
        results.append(item)

    return {"ok": True, "error": None, "result": results}


retrieved = retrieve_documents("How does RAG use retrieved sources?", KNOWLEDGE_BASE)
retrieved


#### 5.2 Grounded answer construction

The answer builder below is deliberately conservative. It does not invent new facts. It either builds an answer from retrieved snippets or says that the available context is insufficient.

In a real RAG system, this step is usually performed by a language model. Even then, the instruction given to the model should be essentially the same as what this function enforces in code:

```text
Use only the retrieved context.
If the context is insufficient, say so.
Show the sources.
Do not invent details.
```

Run the next cell, then compare the answer text with document D003. Every phrase in the answer should be traceable back to a retrieved snippet — that traceability is what "grounded" means in practice.


In [ ]:
def build_grounded_answer(question: str, retrieved_docs: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Build a conservative answer using only retrieved documents."""

    # The empty-retrieval branch is the most important safety feature in
    # this notebook. When no relevant document exists, the correct output
    # is an explicit insufficient-context statement, never a guess.
    if not retrieved_docs:
        return {
            "ok": True,
            "error": None,
            "result": {
                "answer": "The available approved context does not contain enough information to answer this question.",
                "sources": [],
                "limitations": ["No relevant approved document was retrieved."],
            },
        }

    # The answer is a concatenation of retrieved snippets, not a paraphrase.
    # This makes it stilted but perfectly traceable: every sentence exists
    # in a source document. A real model adds fluency, and with it the risk
    # of adding claims -- which is why sources stay attached either way.
    combined_context = " ".join(doc["text"] for doc in retrieved_docs)

    answer = (
        "Based on the retrieved approved context, "
        f"{combined_context}"
    )

    return {
        "ok": True,
        "error": None,
        "result": {
            "answer": answer,
            # Sources include the score so a reader can judge evidence
            # strength, not just evidence presence.
            "sources": [
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "source": doc["source"],
                    "score": doc["score"],
                }
                for doc in retrieved_docs
            ],
            # Limitations are attached even to successful answers, so the
            # reader always knows what the answer is and is not based on.
            "limitations": [
                "The answer is generated only from the retrieved approved snippets.",
                "If the retrieved context is weak or incomplete, the answer may be incomplete.",
            ],
        },
    }


answer = build_grounded_answer("How does RAG use retrieved sources?", retrieved["result"])
answer


#### 5.3 Full local RAG system

Now we combine retrieval and answer construction into one class. This mirrors the chain structure from M04 — a fixed sequence of components, each validating its input — with a retrieval stage added at the front. Notice that `invoke` returns the *whole trace* (question, retrieved documents and answer), not just the final text. Keeping the trace is what makes the next section's inspection possible.


In [ ]:
class LocalRAGSystem:
    """A transparent local RAG system for teaching retrieval and grounding."""

    def __init__(self, knowledge_base: List[Dict[str, Any]]):
        self.knowledge_base = knowledge_base

    def invoke(self, question: str, top_k: int = 3) -> Dict[str, Any]:
        # Each stage can fail independently, and the pipeline stops at the
        # first failure. Passing the error straight through means the caller
        # always learns *which* stage failed and why.
        retrieval = retrieve_documents(question, self.knowledge_base, top_k=top_k)
        if not retrieval["ok"]:
            return retrieval

        answer = build_grounded_answer(question, retrieval["result"])
        if not answer["ok"]:
            return answer

        # Return the full trace, not only the answer text. Inspection of
        # retrieved documents is how you debug a RAG system.
        return {
            "ok": True,
            "error": None,
            "result": {
                "question": question,
                "retrieved_docs": retrieval["result"],
                "answer": answer["result"],
            },
        }


rag = LocalRAGSystem(KNOWLEDGE_BASE)
rag_result = rag.invoke("How does RAG use retrieved sources?")
rag_result


<a id="m05a-inspection"></a>

#### 4.2 Inspecting Retrieval and Grounding

A RAG system should not be evaluated only by reading the final answer. You should inspect:

```text
1. the question,
2. the retrieved documents,
3. the retrieval scores,
4. the answer,
5. the sources,
6. the limitations.
```

The display function below makes those parts visible.

In [ ]:
def display_rag_result(rag_result: Dict[str, Any]) -> None:
    if not rag_result.get("ok"):
        print("ERROR:", rag_result.get("error"))
        return

    result = rag_result["result"]
    print("Question:", result["question"])

    print("\nRetrieved documents:")
    if not result["retrieved_docs"]:
        print("- None")
    for doc in result["retrieved_docs"]:
        print(f"- {doc['doc_id']} | score={doc['score']} | {doc['title']}")
        print(f"  Text: {doc['text']}")

    print("\nAnswer:")
    print(result["answer"]["answer"])

    print("\nSources:")
    if not result["answer"]["sources"]:
        print("- None")
    for source in result["answer"]["sources"]:
        print(f"- {source['doc_id']} | {source['title']} | {source['source']} | score={source['score']}")

    print("\nLimitations:")
    for limitation in result["answer"]["limitations"]:
        print("-", limitation)


display_rag_result(rag_result)

#### 6.1 Interpreting retrieval results

A retrieval result should be judged before the answer is trusted.

<div align="center">

<table>
<thead>
<tr><th><strong>Retrieval outcome</strong></th><th><strong>Interpretation</strong></th><th><strong>What to do</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Relevant document retrieved with high score</td><td>The answer has a useful evidence base.</td><td>Check whether the answer stays within the source.</td></tr>
<tr><td align="left">Only weakly related document retrieved</td><td>The answer may be incomplete or misleading.</td><td>Improve query, document tags or retrieval method.</td></tr>
<tr><td align="left">No document retrieved</td><td>The system lacks approved context.</td><td>Return insufficient-context response.</td></tr>
<tr><td align="left">Wrong document retrieved</td><td>The answer should not be trusted.</td><td>Debug retrieval before changing the answer generator.</td></tr>
</tbody>
</table>

</div>

A common mistake is trying to fix every RAG failure by changing the model. In many cases, the real problem is retrieval: the right context was never provided.

<a id="m05a-optional"></a>

#### 4.3 Optional Embeddings or Real Model Section

This section is optional. The mandatory RAG pipeline already teaches the core architecture. If packages and API access are available, you may later replace lexical retrieval with embeddings, or replace the conservative answer builder with a real model call.

Whichever component you upgrade, the architecture should remain the same:

```text
Question --> [ Retriever ] --> Retrieved context --> [ Prompt ]
                                                         |
                                                         v
                                                   [ Real model ]
                                                         |
                                                         v
                                            Answer with sources
```

Swapping a component must not remove the safety behaviour around it: sources still displayed, insufficient context still reported, no private documents indexed. Do not hard-code API keys. If you do not have a valid API key, simply write:

```text
Skipped: no API key available.
```


In [ ]:
# Optional package installation.
# In Colab, uncomment the line below and run this cell once; the packages
# install into the notebook runtime only. Leave it commented if you are
# skipping the optional section.

# !pip install -q langchain langchain-core langchain-openai


In [ ]:
import os
from getpass import getpass

# Never paste an API key into a code cell: it would be saved with the
# notebook and shared with anyone who sees your submission. The safe
# pattern is to read the key from the environment, and (optionally) to
# prompt for it with getpass, which hides the input and stores it only in
# this session's memory. Uncomment the two lines below to use the prompt.

# if not os.environ.get("OPENAI_API_KEY"):
#     os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY (input hidden): ")

has_openai_key = bool(os.environ.get("OPENAI_API_KEY"))
print("OPENAI_API_KEY found:", has_openai_key)

if not has_openai_key:
    print("Skipped optional real-model section: no API key available.")


In [ ]:
def optional_real_rag_answer(question: str, retrieved_docs: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Optional real-model answer generation from retrieved context."""

    import os

    # Guard clauses first: this function degrades gracefully when the key
    # or the packages are missing, instead of crashing half-way through.
    if not os.environ.get("OPENAI_API_KEY"):
        return {"ok": False, "error": "OPENAI_API_KEY is not set. Skip this optional section.", "result": None}

    try:
        from langchain_openai import ChatOpenAI
        from langchain_core.prompts import ChatPromptTemplate
        from langchain_core.output_parsers import StrOutputParser
    except ImportError as exc:
        return {"ok": False, "error": f"Required packages are not installed: {exc}", "result": None}

    # The retrieved documents become the model's only allowed evidence.
    # Document ids are kept in the context so the model can cite them.
    context = "\n\n".join([f"{doc['doc_id']} {doc['title']}: {doc['text']}" for doc in retrieved_docs])

    # The system prompt encodes the same grounding rules the local answer
    # builder enforced in code. With a real model, rules move into the
    # prompt -- which is exactly why they must also be tested.
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer using only the provided context. If the context is insufficient, say so. Include source document ids."),
        ("human", "Question: {question}\n\nContext:\n{context}")
    ])

    # temperature=0.1 keeps the output close to the evidence; higher values
    # add variety, and with it a higher risk of unsupported additions.
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
    chain = prompt | model | StrOutputParser()

    return {"ok": True, "error": None, "result": chain.invoke({"question": question, "context": context})}


optional_output = optional_real_rag_answer(
    "How does RAG use sources?",
    retrieve_documents("How does RAG use sources?", KNOWLEDGE_BASE)["result"],
)
optional_output


If the optional model call runs, compare it with the conservative answer builder:

```text
Does the real model use only retrieved context?
Does it cite document ids?
Does it add unsupported claims?
Does it behave correctly when context is insufficient?
```

The optional model may produce more fluent text. Fluency is not the same as grounding.

<a id="m05a-testing"></a>

### 5. Testing and Analysis

RAG tests should check both retrieval and answer behaviour, and they should cover the three situations every knowledge system meets:

```text
1. Normal case:              a relevant document exists and is retrieved.
2. Missing-information case: no relevant document exists; the system must
                             say so rather than invent an answer.
3. Failure case:             the input itself is invalid (empty question,
                             top_k = 0); the system must reject it cleanly.
```

The cell below encodes these checks as `assert` statements. If every assertion holds, it prints a success message. If any assertion fails, Python raises `AssertionError` and points at the failing line — read that line to see which behaviour broke, then inspect the corresponding retrieval trace before changing any code.


In [ ]:
test_rag = LocalRAGSystem(KNOWLEDGE_BASE)

# Normal case: a RAG question should retrieve the RAG document (D003)
# and produce an answer with at least one source attached.
rag_case = test_rag.invoke("What is RAG and why should it show sources?")
assert rag_case["ok"] is True
assert len(rag_case["result"]["retrieved_docs"]) >= 1
assert any(doc["doc_id"] == "D003" for doc in rag_case["result"]["retrieved_docs"])
assert len(rag_case["result"]["answer"]["sources"]) >= 1

# Normal case, different topic: checks retrieval is question-driven,
# not simply returning the same document for everything.
tool_case = test_rag.invoke("Why do tool agents need validation and refusal rules?")
assert tool_case["ok"] is True
assert any(doc["doc_id"] == "D004" for doc in tool_case["result"]["retrieved_docs"])

# Normal case, third topic: LangGraph state document.
state_case = test_rag.invoke("How can stateful workflows represent branching?")
assert state_case["ok"] is True
assert any(doc["doc_id"] == "D005" for doc in state_case["result"]["retrieved_docs"])

# Missing-information case: the knowledge base says nothing about exam
# rooms, so the safe behaviour is no sources and an explicit
# insufficient-context answer -- never an invented room number.
weak_case = test_rag.invoke("What is the policy for final exam room allocation?")
assert weak_case["ok"] is True
assert weak_case["result"]["retrieved_docs"] == []
assert weak_case["result"]["answer"]["sources"] == []
assert "does not contain enough information" in weak_case["result"]["answer"]["answer"]

# Failure case: an empty question is malformed input and must be rejected
# with ok=False, not answered and not crashed.
empty = test_rag.invoke("")
assert empty["ok"] is False

# Failure case: top_k=0 is a caller bug; silently returning zero results
# would look like "no relevant documents" and hide the bug.
bad_top_k = test_rag.invoke("RAG", top_k=0)
assert bad_top_k["ok"] is False

print("All M05A mandatory local-RAG tests passed.")


In [ ]:
# Display the full trace for three contrasting questions: two answerable
# from the knowledge base, one deliberately outside it.
for question in [
    "What is RAG and why should it show sources?",
    "Why do tool agents need validation?",
    "What is the final exam room?",
]:
    print("\n==============================")
    display_rag_result(test_rag.invoke(question))


The third example should return an insufficient-context answer. This is correct behaviour. A safe RAG system should not invent exam-room information when no relevant approved document is retrieved.

<a id="m05a-student-tasks"></a>

### 6. Student Tasks

Complete the tasks below in order — each task builds on the previous one. The mandatory local RAG system must run without external API calls. Keep your work in clearly labelled cells so a marker can find each piece of evidence.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run every cell from Setup through Testing and Analysis without modification.</td><td>Confirms your environment reproduces the reference behaviour before you change anything.</td><td>Output showing <code>All M05A mandatory local-RAG tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add one document</td><td>Append one new approved public-style or synthetic document to <code>KNOWLEDGE_BASE</code>, with all five fields: <code>doc_id</code>, <code>title</code>, <code>text</code>, <code>source</code>, <code>tags</code>. Do not use private data.</td><td>RAG quality depends on the knowledge base. Writing a document teaches you what makes content retrievable — especially the role of tags.</td><td>A code cell showing the complete new document.</td></tr>
<tr><td align="left">Task 3: Query your document</td><td>Ask a question whose words overlap your new document, and show the result with <code>display_rag_result</code>.</td><td>Verifies your document is actually retrievable, not just stored. If it is not retrieved, adjust the question or the tags and explain what changed.</td><td>Output showing your <code>doc_id</code> in the retrieved documents and sources.</td></tr>
<tr><td align="left">Task 4: Add tests</td><td>Write at least three <code>assert</code>-based tests: a normal case (your document is retrieved), a missing-information case (an off-topic question returns no sources and the insufficient-context answer), and a failure case (an empty question or <code>top_k=0</code> returns <code>ok=False</code>).</td><td>Normal, missing-information and failure cases are the minimum test set for any RAG system; a system tested only on easy questions will fail silently on hard ones.</td><td>A test cell that runs with all assertions passing.</td></tr>
<tr><td align="left">Task 5: Analyse grounding</td><td>For one answer, identify which source supports it, which claim comes from that source, and whether the answer added anything unsupported.</td><td>Grounding analysis is the habit that catches confident-but-wrong answers, in this lab and in real systems.</td><td>A short grounding paragraph in a markdown cell.</td></tr>
<tr><td align="left">Task 6: Optional real model</td><td>If you have API access, run the optional section safely using the environment-variable pattern. If not, write <code>Skipped: no API key available</code>.</td><td>Comparing a real model's answer with the conservative builder shows that fluency and grounding are different qualities.</td><td>Real model output, or the skipped note.</td></tr>
<tr><td align="left">Task 7: Reflection</td><td>Explain why RAG needs retrieval inspection, source display and insufficient-context handling.</td><td>Being able to justify the architecture matters more than reproducing it.</td><td>150–250 words in a markdown cell.</td></tr>
</tbody>
</table>

</div>


In [ ]:
# Student task starter (Tasks 2 and 3).
#
# Step 1: design one approved public-style or synthetic document.
#         Give the text 2-3 sentences and choose tags that a student
#         question would plausibly contain.
# Step 2: append it to KNOWLEDGE_BASE and rebuild the RAG system.
# Step 3: ask a question that shares words with your document, and check
#         with display_rag_result that YOUR doc_id appears in the sources.
#
# Uncomment and adapt the example below.

# new_doc = {
#     "doc_id": "D006",
#     "title": "Human Review in AI Workflows",
#     "text": "Human review is important when AI systems generate drafts, make recommendations or use tools. Review helps detect unsupported claims and unsafe outputs.",
#     "source": "synthetic public teaching snippet",
#     "tags": ["human_review", "safety", "drafting", "tools"]
# }
#
# KNOWLEDGE_BASE.append(new_doc)
# student_rag = LocalRAGSystem(KNOWLEDGE_BASE)
# display_rag_result(student_rag.invoke("Why is human review important for AI drafts?"))


<a id="m05a-submission"></a>

### 7. Submission and Reflection

**Required submission items**

Submit the completed notebook with:

```text
1. Mandatory baseline test output.
2. Your new knowledge-base document.
3. RAG output showing your new document retrieved.
4. At least three added tests using assert statements.
5. Short grounding analysis.
6. Optional real-model result or skipped note.
7. 150-250 word reflection.
```

**Quality checks**

Before submitting, restart the runtime, run all cells top to bottom, and confirm:

- Every cell runs without errors in a fresh runtime.
- No API key, password or private document appears anywhere in the notebook.
- Your new document is public-style or synthetic teaching content.
- Your three added tests pass, and cover normal, missing-information and failure cases.
- The exam-room question still returns the insufficient-context answer, even after your changes.

**Debugging guide**

- `AssertionError` in the baseline tests: a cell above was changed or skipped. Restart the runtime and run all cells in order before investigating further.
- Your document is never retrieved: print `tokenize(your_question)` and `tokenize(your_document_text)` and look for shared terms. Lexical retrieval needs word overlap — adjust the question wording or the document tags.
- The wrong document is retrieved: check the scores with `score_document`. Another document probably shares more terms with your question; make your question more specific.
- Empty question does not return `ok=False`: you are probably calling `build_grounded_answer` directly instead of going through `rag.invoke`, which is where input validation happens.
- Optional section errors: the API key or packages are missing. That is expected — write the skipped note and move on.

**Reflection questions**

1. What is the difference between retrieval and generation?
2. Why should a RAG answer show sources?
3. What should the system do when no relevant context is retrieved?
4. What is the risk of answering beyond retrieved evidence?
5. How does this prepare for M05B unit-material RAG and M05C LangGraph?

#### Further Readings

- LangChain RAG tutorials: <https://python.langchain.com/docs/tutorials/rag/>
- LangChain retrievers: <https://python.langchain.com/docs/concepts/retrievers/>
- LangChain vector stores: <https://python.langchain.com/docs/concepts/vectorstores/>
- LangGraph documentation: <https://langchain-ai.github.io/langgraph/>
- Public data repository for this unit: <https://github.com/tulip-lab/open-data>
